In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import deepdish as dd
import numpy as np

class BrainMatrixDataset(Dataset):
    def __init__(self, h5_dir):
        self.h5_dir = h5_dir
        self.files = [f for f in os.listdir(h5_dir) if f.endswith('.h5')]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = os.path.join(self.h5_dir, self.files[idx])
        
        # Load your specific format
        data = dd.io.load(file_path)
        corr = data['corr']
        pcorr = data['pcorr']
        label = data['label']
        
        # Stack them to create a 2-channel input!
        # Output shape: (2, 400, 400)
        x = np.stack([corr, pcorr], axis=0)
        
        # Convert to PyTorch tensors
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(label, dtype=torch.float32) # or torch.long for CrossEntropy
        
        return x, y

data_dir = os.path.expanduser('~/Desktop/math_results_braingnn')
# Example usage:
dataset = BrainMatrixDataset(h5_dir=data_dir)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

class BrainMatrixCNN(nn.Module):
    def __init__(self):
        super(BrainMatrixCNN, self).__init__()
        
        # Input shape: (Batch_Size, 2, 200, 200)
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=2, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: (16, 100, 100)
            
            # Block 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: (32, 50, 50)
            
            # Block 3
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=5, stride=5)  # Output: (64, 10, 10)
        )
        
        # Dimensionality after pooling: 64 channels * 10 * 10 spatial size = 6400
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.6), # Extremely high dropout for small dataset
            nn.Linear(6400, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 1) # Output 1 logit for binary classification
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [4]:
# Assuming you have the BrainMatrixDataset class from the previous step
# dataset = BrainMatrixDataset(h5_dir='~/Desktop/math_results_gnn_raw/raw')

# Split into train and validation (e.g., 80% train, 20% val)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)

# Initialize Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainMatrixCNN().to(device)

# BCEWithLogitsLoss is numerically more stable than Sigmoid + BCELoss
criterion = nn.BCEWithLogitsLoss() 

# AdamW applies proper L2 weight decay (critical for preventing overfitting here)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)

# Training Loop
num_epochs = 50

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass (squeeze removes the extra dimension from the 1-node output)
        outputs = model(inputs).squeeze() 
        loss = criterion(outputs, labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
        # Convert logits to binary predictions (threshold at 0)
        predicted = (outputs > 0.0).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = correct_train / total_train
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0.0).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = correct_val / total_val
    
    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")

Epoch [1/50] Train Loss: 1.4968, Train Acc: 0.6042 | Val Loss: 0.6922, Val Acc: 0.5833
Epoch [2/50] Train Loss: 2.0376, Train Acc: 0.3333 | Val Loss: 0.7226, Val Acc: 0.4167
Epoch [3/50] Train Loss: 1.2541, Train Acc: 0.5000 | Val Loss: 0.6908, Val Acc: 0.5833
Epoch [4/50] Train Loss: 1.0081, Train Acc: 0.4792 | Val Loss: 0.6861, Val Acc: 0.5833
Epoch [5/50] Train Loss: 0.9109, Train Acc: 0.6042 | Val Loss: 0.7308, Val Acc: 0.4167
Epoch [6/50] Train Loss: 0.7780, Train Acc: 0.5625 | Val Loss: 0.7027, Val Acc: 0.5833
Epoch [7/50] Train Loss: 0.7034, Train Acc: 0.5625 | Val Loss: 0.7366, Val Acc: 0.5833
Epoch [8/50] Train Loss: 0.6987, Train Acc: 0.6458 | Val Loss: 0.7458, Val Acc: 0.3333
Epoch [9/50] Train Loss: 0.7155, Train Acc: 0.5417 | Val Loss: 0.7594, Val Acc: 0.3333
Epoch [10/50] Train Loss: 0.6797, Train Acc: 0.5208 | Val Loss: 0.7905, Val Acc: 0.3333
Epoch [11/50] Train Loss: 0.6641, Train Acc: 0.5833 | Val Loss: 0.7914, Val Acc: 0.5833
Epoch [12/50] Train Loss: 0.6880, Train A